# 날짜가 같으면 같은 시점의 정보일까요?

이 실습은 이 폴더의 **합성 CSV 네 개**만 사용합니다. 실제 시장의 휴장일이나 누락 원인을 설명하는 자료가 아닙니다.

1. 달력과 가격표를 비교해 주말, 설정 휴장, 거래일 행 누락, 행의 가격 빈값을 구분합니다.
2. 종목·계열·날짜만 맞추는 **의도적으로 잘못된 결합**을 만들고, 판단 이후에 알려진 정보가 붙었는지 확인합니다.
3. 오전 판단에 관측 B를 사용할 수 있을지 예상과 이유를 적습니다.

4. 사용 가능 시각을 기준으로 올바르게 결합하고, 공개 지연 실험과 시간대 변환을 확인합니다.

원본 CSV는 수정하거나 빈 가격을 채우지 않습니다. 결과만 별도 CSV로 저장합니다.

## 준비: 원본 읽기

노트북이 있는 폴더에서 실행하세요. 날짜와 시각은 CSV의 한국 시간대(`+09:00`)를 유지합니다. 화면에 보이는 표만 한국어 열 이름과 짧은 시각 표기를 사용합니다.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 100)
pd.set_option("display.max_rows", 30)
base = Path(".")

In [2]:
calendar = pd.read_csv(base / "market-calendar.csv", parse_dates=["date"])
prices = pd.read_csv(base / "market-observations.csv", parse_dates=["date"])
observations = pd.read_csv(base / "delayed-observations.csv")
decisions = pd.read_csv(base / "decisions.csv")

## 1. 달력과 가격표의 빈자리 구분

가격표에 있는 `SYNTH-A`를 대상으로 달력의 모든 날짜를 남깁니다. 가격이 비었는지만 보면 **행 자체가 없는 경우**와 **행은 있지만 가격이 빈 경우**를 구별할 수 없습니다. 결합 표시(`_merge`)로 행 존재 여부를 따로 확인합니다.

주말과 설정 휴장은 달력의 `is_trading_day`, `calendar_reason`을 근거로 구분합니다. 설정 휴장은 CSV의 ‘예제 휴장’이며 실제 공휴일로 해석하지 않습니다.

In [3]:
price_calendar = calendar.merge(
    prices.loc[prices["symbol"].eq("SYNTH-A"), ["date", "close"]],
    on="date", how="left", indicator=True, validate="one_to_one",
)
price_calendar["row_present"] = price_calendar["_merge"].eq("both")

In [4]:
def classify(row):
    if row["is_trading_day"] == 0:
        return "주말" if row["calendar_reason"] == "주말" else "설정 휴장"
    if not row["row_present"]:
        return "거래일 행 누락"
    if pd.isna(row["close"]):
        return "행의 가격 빈값"
    return "가격 있음"

price_calendar["구분"] = price_calendar.apply(classify, axis=1)

### 표 1. SYNTH-A의 달력과 가격 행 상태

‘가격 표시’의 빈값은 원본 행이 존재할 때만 가격 빈값으로 분류합니다. 행 누락과 가격 빈값의 원인은 이 파일들만으로 알 수 없습니다.

In [5]:
calendar_view = pd.DataFrame({
    "날짜": price_calendar["date"].dt.strftime("%Y-%m-%d"),
    "달력 근거": price_calendar["calendar_reason"],
    "가격 행": price_calendar["row_present"].map({True: "있음", False: "없음"}),
    "가격 표시": price_calendar["close"].map(
        lambda value: "—" if pd.isna(value) else f"{value:g}"
    ),
    "구분": price_calendar["구분"],
})
display(calendar_view.style.hide(axis="index"))

날짜,달력 근거,가격 행,가격 표시,구분
2026-09-01,거래일,있음,100,가격 있음
2026-09-02,거래일,있음,101,가격 있음
2026-09-03,거래일,없음,—,거래일 행 누락
2026-09-04,예제 휴장,없음,—,설정 휴장
2026-09-05,주말,없음,—,주말
2026-09-06,주말,없음,—,주말
2026-09-07,거래일,있음,106,가격 있음
2026-09-08,거래일,있음,107,가격 있음
2026-09-09,거래일,있음,—,행의 가격 빈값
2026-09-10,거래일,있음,109,가격 있음


## 2. 의도적으로 잘못된 결합: 종목·계열·날짜만 일치시키기

`observed_at`은 관측 시각, `published_at`은 공개 시각, `known_at`은 알려진 시각, `retrieved_at`은 가져온 시각입니다. `decision_at`은 판단 시각입니다. 이 실습에서는 **알려진 시각이 판단 시각보다 늦은지**를 검사합니다. `known_at`이 비어 있으면 가용 여부를 확정하지 않습니다.

잘못된 예에서는 판단 시각의 날짜를 관측 날짜에 맞춥니다. 종목과 계열도 같아야 하지만, 날짜 안의 시간 순서는 결합 조건에서 무시합니다. 아래 코드는 잘못된 결과를 관찰하기 위한 것으로 실제 분석에 사용하면 안 됩니다.

In [6]:
# 원본 문자열에서 시간대 유무부터 검사합니다. UTC 변환으로 시간대를 추정하지 않습니다.
TIME_COLUMNS = ["observed_at", "published_at", "known_at", "retrieved_at"]

def checked_times(frame, columns, id_column):
    result = frame.copy(deep=True)
    for column in columns:
        converted = []
        for row_id, value in zip(result[id_column], result[column]):
            if pd.isna(value) or str(value).strip() == "":
                converted.append(pd.NaT)
                continue
            try:
                stamp = pd.Timestamp(value)
            except (ValueError, TypeError) as exc:
                raise ValueError(f"{row_id}: {column} 시각 형식 오류") from exc
            if pd.isna(stamp) or stamp.tzinfo is None or stamp.utcoffset() is None:
                raise ValueError(f"{row_id}: {column} 시간대 없는 시각 또는 잘못된 시각")
            converted.append(stamp.tz_convert("UTC"))
        result[column] = pd.Series(converted, index=result.index, dtype="datetime64[ns, UTC]")
    return result

def validate_inputs(observation_input, decision_input):
    obs = checked_times(observation_input, TIME_COLUMNS, "observation_id")
    dec = checked_times(decision_input, ["decision_at"], "decision_id")
    for frame, id_column in [(obs, "observation_id"), (dec, "decision_id")]:
        if frame[[id_column, "symbol", "series"]].isna().any().any():
            raise ValueError(f"{id_column}: ID 또는 종목·계열 누락")
        if frame[id_column].duplicated().any():
            raise ValueError(f"{id_column}: ID 중복")
    if obs["observed_at"].isna().any() or dec["decision_at"].isna().any():
        raise ValueError("관측 시각 또는 판단 시각 누락")
    # 중간 시각이 미상이어도 확인 가능한 두 시각의 역전은 허용하지 않습니다.
    for earlier, later in [("observed_at", "published_at"),
                           ("published_at", "known_at"), ("observed_at", "known_at")]:
        bad = obs[earlier].notna() & obs[later].notna() & obs[earlier].gt(obs[later])
        if bad.any():
            raise ValueError(f"{earlier} → {later} 역전: {obs.loc[bad, 'observation_id'].tolist()}")
    known = obs.loc[obs["known_at"].notna()]
    conflict = known.duplicated(["symbol", "series", "known_at"], keep=False)
    if conflict.any():
        raise ValueError(f"known_at 충돌: {known.loc[conflict, 'observation_id'].tolist()}")
    return obs, dec

# 기존 의도적 오류도 잘못된 입력을 통과시키지 않도록 먼저 검증합니다.
validate_inputs(observations, decisions)

(  observation_id   symbol   series               observed_at              published_at  \
 0              A  SYNTH-A   signal 2026-09-07 06:00:00+00:00 2026-09-07 09:00:00+00:00   
 1              B  SYNTH-A   signal 2026-09-08 06:00:00+00:00 2026-09-08 09:00:00+00:00   
 2              C  SYNTH-A  unknown 2026-09-08 06:00:00+00:00                       NaT   
 3              F  SYNTH-A    first 2026-09-08 06:00:00+00:00 2026-09-08 09:00:00+00:00   
 4              Z  SYNTH-Z   signal 2026-09-07 22:00:00+00:00 2026-09-07 23:00:00+00:00   
 
                    known_at              retrieved_at  value  
 0 2026-09-07 09:00:00+00:00 2026-09-13 03:00:00+00:00     10  
 1 2026-09-08 09:00:00+00:00 2026-09-13 03:00:00+00:00     20  
 2                       NaT 2026-09-13 03:00:00+00:00     30  
 3 2026-09-08 09:00:00+00:00 2026-09-13 03:00:00+00:00     40  
 4 2026-09-07 23:00:00+00:00 2026-09-13 03:00:00+00:00    900  ,
   decision_id   symbol   series               decision_at
 0      

In [7]:
for column in ["observed_at", "published_at", "known_at", "retrieved_at"]:
    observations[column] = pd.to_datetime(observations[column])
decisions["decision_at"] = pd.to_datetime(decisions["decision_at"])
observations["join_date"] = observations["observed_at"].dt.date
decisions["join_date"] = decisions["decision_at"].dt.date

### 표 2. 날짜 결합 전 관측 정보

표의 모든 시각은 한국 시간입니다. `미상`은 CSV의 빈값이며 다른 시각으로 대체하지 않습니다. 공개·수집 시각은 데이터에 보존하되 이 표에서는 핵심 비교 열만 표시합니다.

In [8]:
def short_time(series):
    return series.dt.strftime("%m-%d %H:%M").fillna("미상")

observation_view = observations[["observation_id", "symbol", "series", "value"]].rename(
    columns={"observation_id": "관측", "symbol": "종목", "series": "계열", "value": "값"}
)
observation_view["관측 시각"] = short_time(observations["observed_at"])
observation_view["알려진 시각"] = short_time(observations["known_at"])
display(observation_view.style.hide(axis="index"))

관측,종목,계열,값,관측 시각,알려진 시각
A,SYNTH-A,signal,10,09-07 15:00,09-07 18:00
B,SYNTH-A,signal,20,09-08 15:00,09-08 18:00
C,SYNTH-A,unknown,30,09-08 15:00,미상
F,SYNTH-A,first,40,09-08 15:00,09-08 18:00
Z,SYNTH-Z,signal,900,09-08 07:00,09-08 08:00


In [9]:
# 의도적 오류: known_at과 decision_at의 선후 관계를 결합 조건에서 제외합니다.
wrong_join = decisions.merge(
    observations,
    on=["symbol", "series", "join_date"],
    how="left", indicator=True, validate="many_to_one",
)
late = wrong_join["known_at"].gt(wrong_join["decision_at"])
matched = wrong_join["_merge"].eq("both")

In [10]:
wrong_join["점검"] = "늦지 않음"
wrong_join.loc[late, "점검"] = "판단 이후 알려짐"
wrong_join.loc[matched & wrong_join["known_at"].isna(), "점검"] = "알려진 시각 미상"
wrong_join.loc[~matched, "점검"] = "같은 날짜 관측 없음"

wrong_view = pd.DataFrame({
    "판단": wrong_join["decision_id"],
    "관측": wrong_join["observation_id"].fillna("—"),
    "판단 시각": short_time(wrong_join["decision_at"]),
    "알려진 시각": short_time(wrong_join["known_at"]),
    "점검": wrong_join["점검"],
})
wrong_view.loc[~matched, "알려진 시각"] = "—"

### 표 3. 잘못된 날짜 결합 결과와 시간 점검

‘늦지 않음’은 이 표의 `known_at > decision_at` 검사에 걸리지 않았다는 뜻입니다. 올바른 결합의 정답을 뜻하지 않습니다. 같은 날짜 관측이 없다는 표시 역시 사용할 정보가 전혀 없다는 뜻은 아닙니다.

In [11]:
display(wrong_view.style.hide(axis="index"))

판단,관측,판단 시각,알려진 시각,점검
d1,B,09-08 09:00,09-08 18:00,판단 이후 알려짐
d2,B,09-08 19:00,09-08 18:00,늦지 않음
d3,F,09-08 09:00,09-08 18:00,판단 이후 알려짐
d4,C,09-08 19:00,미상,알려진 시각 미상
d5,B,09-08 18:00,09-08 18:00,늦지 않음
d6,Z,09-08 09:00,09-08 08:00,늦지 않음
d7,—,09-06 09:00,—,같은 날짜 관측 없음


### 표 4. 판단 시각보다 늦게 알려진 행

아래 표는 잘못 붙은 행 중 `known_at > decision_at`인 행만 골라냅니다. ‘얼마나 늦음’은 두 시각의 차이일 뿐, 지연의 원인을 설명하지 않습니다. 같은 종목·계열·날짜라는 조건만으로는 판단 당시 알려져 있었음을 보장할 수 없습니다.

In [12]:
late_view = wrong_view.loc[late].drop(columns="점검").copy()
late_view["얼마나 늦음"] = (
    (wrong_join.loc[late, "known_at"] - wrong_join.loc[late, "decision_at"])
    .dt.total_seconds().div(3600).map(lambda hours: f"{hours:g}시간")
)
display(late_view.style.hide(axis="index"))

판단,관측,판단 시각,알려진 시각,얼마나 늦음
d1,B,09-08 09:00,09-08 18:00,9시간
d3,F,09-08 09:00,09-08 18:00,9시간


## 3. 학습자 예상: 오전 판단에 B를 사용할 수 있을까요?

2026년 9월 8일 오전 09:00의 `d1` 판단(`SYNTH-A`, `signal`)을 생각해 보세요. 위의 관측표와 잘못된 결합 결과를 근거로 아래 Markdown 셀을 직접 채우세요. 예상을 먼저 적은 뒤 아래 올바른 결합 결과와 비교하세요.

### 내 예상과 이유

- **오전 판단에서 B를 사용할 수 있는가?**: [예상 작성]
- **비교한 시각과 그 의미**: [판단 시각, 관측 시각, 알려진 시각을 비교해 작성]
- **그렇게 예상한 이유**: [같은 날짜로 붙은 결과만 믿어도 되는지 설명]
- **아직 확인해야 할 점**: [남은 질문 작성]

## 4. 올바른 시각 결합

같은 종목·계열에서 `known_at <= decision_at`인 관측 중 `known_at`이 가장 늦은 것을 선택합니다. 관측 날짜가 전날이어도 후보이며, 판단과 정확히 같은 시각도 포함합니다. 날짜나 수집 시각으로 가용 시각을 대신하지 않습니다.

모든 판단 행과 순서를 보존합니다. 미선택 값은 공란으로 저장하고 사유를 별도 열에 둡니다. `known_at` 미확인 관측은 별도 제외 표에 원본 열·ID·사유를 보존하고, 해당 종목·계열의 판단 결과에도 제외 ID와 사유를 기록합니다. 화면은 한국 시간, 저장 파일은 UTC 오프셋을 포함합니다.

In [13]:
# 원본 파일을 다시 읽어 복사본에서만 작업합니다.
raw_observations = pd.read_csv(base / "delayed-observations.csv")
raw_decisions = pd.read_csv(base / "decisions.csv")

def align_observations(observation_input, decision_input):
    obs, dec = validate_inputs(observation_input, decision_input)
    excluded = obs.loc[obs["known_at"].isna()].copy()
    excluded["exclusion_reason"] = "known_at 미확인: 추정하지 않고 제외"
    usable = obs.loc[obs["known_at"].notna()]
    rows = []
    for _, decision in dec.iterrows():
        same = usable.loc[usable["symbol"].eq(decision["symbol"])
                          & usable["series"].eq(decision["series"])]
        candidates = same.loc[same["known_at"].le(decision["decision_at"])]
        unknown = excluded.loc[excluded["symbol"].eq(decision["symbol"])
                               & excluded["series"].eq(decision["series"])]
        row = decision.to_dict()
        row.update(selected_id=pd.NA, value=pd.NA, known_at=pd.NaT,
                   excluded_ids="|".join(unknown["observation_id"].astype(str)),
                   exclusion_reason="known_at 미확인: 추정하지 않고 제외" if len(unknown) else "")
        if len(candidates):
            selected = candidates.loc[candidates["known_at"].idxmax()]
            row.update(selected_id=selected["observation_id"], value=selected["value"],
                       known_at=selected["known_at"], reason="조건 충족: 가장 늦은 사용 가능 관측")
        elif len(same):
            row["reason"] = "판단 시각까지 사용 가능한 관측 없음"
        elif len(unknown):
            row["reason"] = "known_at 미확인 관측만 있어 선택 불가"
        else:
            row["reason"] = "같은 종목·계열의 관측 없음"
        rows.append(row)
    aligned = pd.DataFrame(rows, columns=[
        "decision_id", "symbol", "series", "decision_at", "selected_id", "value",
        "known_at", "reason", "excluded_ids", "exclusion_reason",
    ])
    aligned["known_at"] = pd.to_datetime(aligned["known_at"], utc=True)
    aligned["decision_at"] = pd.to_datetime(aligned["decision_at"], utc=True)
    return aligned, excluded

aligned, excluded_observations = align_observations(raw_observations, raw_decisions)

def result_view(result):
    return pd.DataFrame({
        "판단 ID": result["decision_id"],
        "선택 ID": result["selected_id"].fillna(""),
        "값": result["value"].fillna(""),
        "사용 가능 시각 (KST)": result["known_at"].dt.tz_convert("Asia/Seoul").dt.strftime("%Y-%m-%d %H:%M").fillna(""),
        "판단 시각 (KST)": result["decision_at"].dt.tz_convert("Asia/Seoul").dt.strftime("%Y-%m-%d %H:%M"),
        "사유": result["reason"],
    })

display(result_view(aligned).style.hide(axis="index"))
display(excluded_observations[["observation_id", "symbol", "series", "exclusion_reason"]]
        .rename(columns={"observation_id": "제외 ID", "symbol": "종목", "series": "계열",
                         "exclusion_reason": "제외 사유"}).style.hide(axis="index"))

판단 ID,선택 ID,값,사용 가능 시각 (KST),판단 시각 (KST),사유
d1,A,10,2026-09-07 18:00,2026-09-08 09:00,조건 충족: 가장 늦은 사용 가능 관측
d2,B,20,2026-09-08 18:00,2026-09-08 19:00,조건 충족: 가장 늦은 사용 가능 관측
d3,,,,2026-09-08 09:00,판단 시각까지 사용 가능한 관측 없음
d4,,,,2026-09-08 19:00,known_at 미확인 관측만 있어 선택 불가
d5,B,20,2026-09-08 18:00,2026-09-08 18:00,조건 충족: 가장 늦은 사용 가능 관측
d6,Z,900,2026-09-08 08:00,2026-09-08 09:00,조건 충족: 가장 늦은 사용 가능 관측
d7,,,,2026-09-06 09:00,판단 시각까지 사용 가능한 관측 없음


제외 ID,종목,계열,제외 사유
C,SYNTH-A,unknown,known_at 미확인: 추정하지 않고 제외


In [14]:
future_count = int(aligned["known_at"].gt(aligned["decision_at"]).sum())
assert future_count == 0
assert len(aligned) == len(raw_decisions)
assert aligned["decision_id"].tolist() == raw_decisions["decision_id"].tolist()
assert aligned["selected_id"].fillna("").tolist() == ["A", "B", "", "", "B", "Z", ""]
assert excluded_observations["observation_id"].tolist() == ["C"]
assert aligned.loc[aligned["decision_id"].eq("d4"), "excluded_ids"].item() == "C"
aligned.to_csv(base / "aligned-observations.csv", index=False, na_rep="")
display(pd.DataFrame({"검사": ["미래 매칭", "판단 행 보존"],
                      "결과": [f"{future_count}건", f"{len(aligned)}/{len(raw_decisions)}행, ID·순서 일치"]})
        .style.hide(axis="index"))

검사,결과
미래 매칭,0건
판단 행 보존,"7/7행, ID·순서 일치"


## 5. 공개 지연 실험 — 실행 전 예상

B의 `published_at`과 `known_at`을 모두 2026-09-08 20:00+09:00으로 옮기면, 같은 날 19시와 21시 판단은 어떤 관측을 선택할까요? 아래 예상 칸을 먼저 작성하세요. 원본 시나리오와 변경 시나리오의 실제 비교는 다음 코드 셀에서 실행합니다.

| 판단 시각 | 변경 전 선택 ID 예상 | 변경 후 선택 ID 예상 | 이유 |
|---|---|---|---|
| 19:00+09:00 |  |  |  |
| 21:00+09:00 |  |  |  |

In [15]:
experiment_decisions = pd.DataFrame({
    "decision_id": ["experiment-19", "experiment-21"],
    "symbol": ["SYNTH-A", "SYNTH-A"], "series": ["signal", "signal"],
    "decision_at": ["2026-09-08T19:00:00+09:00", "2026-09-08T21:00:00+09:00"],
})
shifted = raw_observations.copy(deep=True)
shifted.loc[shifted["observation_id"].eq("B"), ["published_at", "known_at"]] = "2026-09-08T20:00:00+09:00"
before, _ = align_observations(raw_observations, experiment_decisions)
after, _ = align_observations(shifted, experiment_decisions)
comparison = pd.concat([
    result_view(before).assign(시나리오="변경 전"),
    result_view(after).assign(시나리오="B 공개·사용 가능 시각 20시"),
], ignore_index=True)
display(comparison[["시나리오", "판단 ID", "선택 ID", "값", "사용 가능 시각 (KST)", "판단 시각 (KST)", "사유"]]
        .style.hide(axis="index"))
assert before["selected_id"].tolist() == ["B", "B"]
assert after["selected_id"].tolist() == ["A", "B"]
for result in [before, after]:
    assert not result["known_at"].gt(result["decision_at"]).any()
    assert result["decision_id"].tolist() == experiment_decisions["decision_id"].tolist()
pd.testing.assert_frame_equal(raw_observations, pd.read_csv(base / "delayed-observations.csv"))
pd.testing.assert_frame_equal(raw_decisions, pd.read_csv(base / "decisions.csv"))

시나리오,판단 ID,선택 ID,값,사용 가능 시각 (KST),판단 시각 (KST),사유
변경 전,experiment-19,B,20,2026-09-08 18:00,2026-09-08 19:00,조건 충족: 가장 늦은 사용 가능 관측
변경 전,experiment-21,B,20,2026-09-08 18:00,2026-09-08 21:00,조건 충족: 가장 늦은 사용 가능 관측
B 공개·사용 가능 시각 20시,experiment-19,A,10,2026-09-07 18:00,2026-09-08 19:00,조건 충족: 가장 늦은 사용 가능 관측
B 공개·사용 가능 시각 20시,experiment-21,B,20,2026-09-08 20:00,2026-09-08 21:00,조건 충족: 가장 늦은 사용 가능 관측


## 6. 같은 순간을 다른 시간대로 표현하기

같은 날짜의 `09:00+09:00`과 `00:00Z`를 실제로 UTC로 변환하고 비교합니다. 시간대 없는 입력에 UTC를 임의로 붙이는 것과는 다릅니다.

In [16]:
kst_stamp = pd.Timestamp("2026-09-08T09:00:00+09:00")
utc_stamp = pd.Timestamp("2026-09-08T00:00:00Z")
converted = kst_stamp.tz_convert("UTC")
assert converted == utc_stamp
assert converted.value == utc_stamp.value
display(pd.DataFrame({"입력": [kst_stamp.isoformat(), utc_stamp.isoformat()],
                      "UTC 변환": [converted.isoformat(), utc_stamp.tz_convert("UTC").isoformat()],
                      "같은 순간": [converted == utc_stamp, converted.value == utc_stamp.value]})
        .style.hide(axis="index"))

입력,UTC 변환,같은 순간
2026-09-08T09:00:00+09:00,2026-09-08T00:00:00+00:00,True
2026-09-08T00:00:00+00:00,2026-09-08T00:00:00+00:00,True


## 7. 잘못된 입력은 결합 전에 중단되는지 검사

시간대 누락, 시각 역전, 같은 종목·계열의 동일한 사용 가능 시각을 복사본에 주입합니다. 충돌은 UTC 변환 뒤 검사하므로 표기가 다른 같은 순간도 잡힙니다.

In [17]:
def must_reject(obs, dec, expected):
    try:
        align_observations(obs, dec)
    except ValueError as exc:
        assert expected in str(exc), str(exc)
    else:
        raise AssertionError(f"중단하지 않음: {expected}")

for column in TIME_COLUMNS:
    invalid = raw_observations.copy(deep=True)
    invalid.loc[0, column] = "2026-09-07T15:00:00"
    must_reject(invalid, raw_decisions, "시간대 없는")
invalid_decisions = raw_decisions.copy(deep=True)
invalid_decisions.loc[0, "decision_at"] = "2026-09-08T09:00:00"
must_reject(raw_observations, invalid_decisions, "시간대 없는")
for column, stamp in [("published_at", "2026-09-07T14:00:00+09:00"),
                       ("known_at", "2026-09-07T17:00:00+09:00")]:
    invalid = raw_observations.copy(deep=True)
    invalid.loc[0, column] = stamp
    must_reject(invalid, raw_decisions, "역전")
invalid = raw_observations.copy(deep=True)
invalid.loc[0, ["published_at", "known_at"]] = "2026-09-08T09:00:00Z"
must_reject(invalid, raw_decisions, "known_at 충돌")
print("시간대 누락·시각 역전·동일 순간 충돌: 모두 오류로 중단 확인")

시간대 누락·시각 역전·동일 순간 충돌: 모두 오류로 중단 확인
